# Modelo Ensemble — combinando los modelos entrenados
## TFG: Predicción de Tráfico Urbano (M30 + URB)

**Sin reentrenamiento.** Este notebook carga los pesos de los 5 modelos PyTorch que ya entrenamos (BiLSTM v2, GRU, Transformer, BiGRU, Transformer grande), predice el test con cada uno y combina esas predicciones para obtener un *ensemble* que normalmente mejora a cualquier modelo individual.

---

### ¿Qué es un ensemble y por qué funciona?

Cada modelo aprende una función diferente de los datos y tiene sus propios **sesgos**: uno tiende a sobreestimar la intensidad en hora punta, otro infraestima la ocupación los fines de semana, etc. Cuando los errores **no están perfectamente correlacionados**, al **promediar** las predicciones de varios modelos esos errores tienden a cancelarse y queda la parte común correcta. El resultado: menos varianza y, casi siempre, mejor MAE/RMSE.

Cuanto más **diversos** sean los modelos (arquitecturas distintas, datos de entrenamiento ligeramente distintos…), mejor funciona el ensemble. Promediar dos modelos casi idénticos apenas aporta; promediar uno recurrente con uno de atención sí.

### Estrategias que probamos aquí

| Estrategia | Cómo se combina |
|---|---|
| **Mean (top-2)** | Media de los 2 mejores por val_loss (BiLSTM + BiGRU) |
| **Mean (top-3)** | Media de los 3 mejores (BiLSTM + BiGRU + Transformer grande) |
| **Mean (all 5)** | Media de los 5 modelos |
| **Weighted (inv val_loss)** | Media ponderada: pesa más el que mejor val_loss tuvo |
| **Median (all 5)** | Mediana — robusta a outliers de un modelo concreto |

Todas usan **el mismo pipeline de datos** (caché NVMe, 22 features, ventana 12 pasos) y predicen sobre **el mismo split de test**, así la comparativa con los modelos individuales es directa.

> Importante: el LSTM v1 (`MMMModel.ipynb`) está en Keras `.keras`, no en PyTorch `.pt`, así que no puedo cargar sus pesos aquí. Lo dejamos fuera del ensemble (su MAE/RMSE sí aparece en la comparativa final, leído de su JSON).

In [ ]:
# 0. Imports y configuración
import os, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

TARGET_COLS = ["intensidad_trafico", "ocupacion", "carga"]
SEQ_LEN     = 12
SEQ_STEPS   = list(range(SEQ_LEN, 0, -1))
N_CTX       = 19
N_FEATURES  = 3 + N_CTX                       # 22

OUT_DIR_PLOTS = "SALIDAS modelo Ensemble"
os.makedirs(OUT_DIR_PLOTS, exist_ok=True)

print(f"PyTorch {torch.__version__}")
print(f"Dispositivo: {DEVICE}", "(AMD ROCm)" if DEVICE.type=="cuda" else "")

## 1. Carga del test y de los estadísticos de normalización

Releo la caché `.npy` en el NVMe (la misma que usaron los notebooks de entrenamiento). Necesito:
- **`lag_test`, `ctx_test`** (mmap, casi sin RAM) — las entradas del test.
- **`y_test`** — las etiquetas reales (sin normalizar) para calcular MAE/RMSE.
- **`X_MEAN`, `X_STD`** — estadísticos de normalización de las 22 features, calculados desde `train` (idéntico a los notebooks de entrenamiento).
- **`scaler_y`** — `StandardScaler` ajustado sobre `y_train`, para invertir las predicciones a unidades reales.

Es crítico usar **exactamente los mismos estadísticos** que en entrenamiento; si normalizo distinto, las predicciones quedan descalibradas.

In [ ]:
CACHE_DIR = os.path.expanduser("~/.cache/prediccion-trafico/seq_cache")
with open(f"{CACHE_DIR}/counts.json") as f: counts = json.load(f)
def _load(name, split, ram=False):
    a = np.load(f"{CACHE_DIR}/{name}_{split}.npy", mmap_mode="r")[:counts[split]]
    return np.array(a) if ram else a

# Train (solo para los estadísticos de normalización)
lag_train = _load("lag", "train")
ctx_train = _load("ctx", "train")
y_train   = _load("y",   "train", ram=True)

# Estadísticos X (idéntico al cálculo que hace cada notebook de entrenamiento)
mean_ = np.empty(N_FEATURES, dtype="float32")
std_  = np.empty(N_FEATURES, dtype="float32")
mean_[:3] = lag_train.mean(axis=(0, 1)); std_[:3] = lag_train.std(axis=(0, 1))
mean_[3:] = ctx_train.mean(axis=0);      std_[3:] = ctx_train.std(axis=0)
std_ = np.where(std_ == 0, 1.0, std_).astype("float32")
X_MEAN = torch.tensor(mean_, device=DEVICE)
X_STD  = torch.tensor(std_,  device=DEVICE)

# scaler_y desde y_train
scaler_y = StandardScaler().fit(y_train)

# Test (lo que vamos a predecir)
lag_test = _load("lag", "test")
ctx_test = _load("ctx", "test")
y_test   = _load("y",   "test", ram=True)
test_df  = pd.read_parquet(f"{CACHE_DIR}/meta_test.parquet")

# Liberar arrays de train (ya no hacen falta tras calcular stats)
del lag_train, ctx_train, y_train; gc.collect()

print(f"test: {lag_test.shape[0]:,} filas")
print(f"X_MEAN[:3]={mean_[:3].round(3)}  scaler_y.mean_={scaler_y.mean_.round(3)}")

## 2. Definición de los 5 modelos entrenados

Para cargar un checkpoint `.pt` (que solo contiene el `state_dict` — los pesos) necesito **reconstruir la misma arquitectura** y luego hacer `load_state_dict`. Pego aquí las 4 clases tal cual están en sus notebooks originales:

- **`BiLSTMv2`** — BiLSTM(128) → BiLSTM(64) → Dense (de `Modelo2.0.ipynb`)
- **`GRUNet`** — GRU(128) → GRU(64) unidireccional (de `Modelo_GRU.ipynb`)
- **`BiGRUNet`** — GRU(128, bi) → GRU(64, bi) (de `Modelo_BiGRU.ipynb`)
- **`TrafficTransformer`** — encoder con auto-atención (sirve para el pequeño y el grande, solo cambian los hiperparámetros)

In [ ]:
class BiLSTMv2(nn.Module):
    def __init__(self, n_features, n_targets=3, dropout=0.3):
        super().__init__()
        self.lstm1  = nn.LSTM(n_features, 128, batch_first=True, bidirectional=True)
        self.drop1  = nn.Dropout(dropout)
        self.lstm2  = nn.LSTM(2*128, 64, batch_first=True, bidirectional=True)
        self.drop2  = nn.Dropout(dropout)
        self.dense1 = nn.Linear(2*64, 64); self.dense2 = nn.Linear(64, 32)
        self.out    = nn.Linear(32, n_targets); self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.lstm1(x); x = self.drop1(x)
        _, (h, _) = self.lstm2(x)
        x = torch.cat([h[0], h[1]], dim=1); x = self.drop2(x)
        x = self.act(self.dense1(x)); x = self.act(self.dense2(x))
        return self.out(x)

class GRUNet(nn.Module):
    def __init__(self, n_features, n_targets=3, dropout=0.3):
        super().__init__()
        self.gru1   = nn.GRU(n_features, 128, batch_first=True)
        self.drop1  = nn.Dropout(dropout)
        self.gru2   = nn.GRU(128, 64, batch_first=True)
        self.drop2  = nn.Dropout(dropout)
        self.dense1 = nn.Linear(64, 64); self.dense2 = nn.Linear(64, 32)
        self.out    = nn.Linear(32, n_targets); self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.gru1(x); x = self.drop1(x)
        _, h = self.gru2(x); x = h[-1]; x = self.drop2(x)
        x = self.act(self.dense1(x)); x = self.act(self.dense2(x))
        return self.out(x)

class BiGRUNet(nn.Module):
    def __init__(self, n_features, n_targets=3, dropout=0.3):
        super().__init__()
        self.gru1   = nn.GRU(n_features, 128, batch_first=True, bidirectional=True)
        self.drop1  = nn.Dropout(dropout)
        self.gru2   = nn.GRU(2*128, 64, batch_first=True, bidirectional=True)
        self.drop2  = nn.Dropout(dropout)
        self.dense1 = nn.Linear(2*64, 64); self.dense2 = nn.Linear(64, 32)
        self.out    = nn.Linear(32, n_targets); self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.gru1(x); x = self.drop1(x)
        _, h = self.gru2(x)
        x = torch.cat([h[0], h[1]], dim=1); x = self.drop2(x)
        x = self.act(self.dense1(x)); x = self.act(self.dense2(x))
        return self.out(x)

class PositionalEmbedding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)
    def forward(self, x): return x + self.pos

class TrafficTransformer(nn.Module):
    def __init__(self, n_features, n_targets=3, d_model=64, nhead=4,
                 num_layers=3, dim_ff=128, dropout=0.3, seq_len=12):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_emb    = PositionalEmbedding(seq_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model); self.drop = nn.Dropout(dropout)
        self.dense1  = nn.Linear(d_model, 64); self.dense2 = nn.Linear(64, 32)
        self.out     = nn.Linear(32, n_targets); self.act = nn.ReLU()
    def forward(self, x):
        x = self.pos_emb(self.input_proj(x))
        x = self.encoder(x); x = self.norm(x).mean(dim=1); x = self.drop(x)
        x = self.act(self.dense1(x)); x = self.act(self.dense2(x))
        return self.out(x)

print("4 clases de modelo definidas (Transformer cubre el pequeño y el grande con distintos kwargs)")

## 3. Inferencia: cargar pesos y predecir el test, modelo a modelo

Para cada modelo:
1. Instancio la clase con los **kwargs correctos** (importante: el Transformer grande usa `d_model=128, nhead=8, num_layers=4, dim_ff=256`).
2. Cargo el `state_dict` desde su `*_best.pt` (mejores pesos del entrenamiento).
3. Predigo el test en lotes con `autocast(fp16)` en GPU.
4. Invierto la normalización con `scaler_y.inverse_transform` para tener las predicciones en **unidades reales**.

Las predicciones de cada modelo se guardan en `preds[nombre]` con shape `(n_test, 3)`.

In [ ]:
BATCH = 4096

def assemble(lag_b, ctx_b):
    """Ensambla (B, 12, 22) en GPU y normaliza con X_MEAN/X_STD."""
    B = lag_b.size(0)
    seq = torch.empty((B, SEQ_LEN, N_FEATURES), device=lag_b.device)
    seq[:, :, :3] = lag_b
    seq[:, :, 3:] = ctx_b.unsqueeze(1).expand(-1, SEQ_LEN, -1)
    return (seq - X_MEAN) / X_STD

@torch.no_grad()
def predict_test(model):
    model.eval()
    ds = TensorDataset(torch.from_numpy(lag_test), torch.from_numpy(ctx_test))
    ld = DataLoader(ds, batch_size=BATCH, shuffle=False, pin_memory=USE_AMP)
    out = []
    for lag_b, ctx_b in ld:
        lag_b = lag_b.to(DEVICE, non_blocking=True)
        ctx_b = ctx_b.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            y = model(assemble(lag_b, ctx_b))
        out.append(y.float().cpu().numpy())
    y_norm = np.concatenate(out, axis=0)
    return scaler_y.inverse_transform(y_norm).astype("float32")   # unidades reales

# Catálogo de modelos a cargar: (nombre, clase, kwargs, ruta del checkpoint)
MODELS = [
    ("BiLSTM v2",     BiLSTMv2,           dict(n_features=N_FEATURES),
     "SALIDAS modelo 2/lstm_v2_best.pt"),
    ("GRU",           GRUNet,             dict(n_features=N_FEATURES, dropout=0.4),
     "SALIDAS modelo GRU/gru_best.pt"),
    ("Transformer",   TrafficTransformer, dict(n_features=N_FEATURES, seq_len=SEQ_LEN, dropout=0.4),
     "SALIDAS modelo Transformer/transformer_best.pt"),
    ("BiGRU",         BiGRUNet,           dict(n_features=N_FEATURES),
     "SALIDAS modelo BiGRU/bigru_best.pt"),
    ("Transf.Grande", TrafficTransformer, dict(n_features=N_FEATURES, seq_len=SEQ_LEN,
                                               d_model=128, nhead=8, num_layers=4, dim_ff=256),
     "SALIDAS modelo Transformer Grande/transformer_grande_best.pt"),
]

preds = {}
for name, Cls, kw, ckpt in MODELS:
    if not os.path.exists(ckpt):
        print(f"  [SKIP] {name}: no existe {ckpt}"); continue
    m = Cls(**kw).to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    p = predict_test(m)
    preds[name] = p
    print(f"  {name:14s} OK  -> preds shape {p.shape}")
    del m; torch.cuda.empty_cache() if DEVICE.type=="cuda" else None

print(f"\n{len(preds)} modelos cargados y predicciones generadas.")

## 4. Métricas individuales (sanity check)

Antes de combinar nada, calculo MAE y RMSE de cada modelo sobre el test. Esto debe coincidir (o estar muy cerca) de los números guardados en cada `metricas_*.json` durante el entrenamiento — si no coincide, hay algo mal en la normalización o en la carga del checkpoint.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def metrics_of(y_pred):
    return {t: {"MAE": float(mean_absolute_error(y_test[:, i], y_pred[:, i])),
                "RMSE": rmse(y_test[:, i], y_pred[:, i])}
            for i, t in enumerate(TARGET_COLS)}

indiv = {name: metrics_of(p) for name, p in preds.items()}

print("MÉTRICAS INDIVIDUALES — TEST")
print("="*70)
print(f"  {'Modelo':<16}" + "".join(f"{t:>20}" for t in TARGET_COLS))
print(f"  {'':<16}" + "  MAE / RMSE       " * 3)
for name, mm in indiv.items():
    row = f"  {name:<16}"
    for t in TARGET_COLS: row += f"   {mm[t]['MAE']:6.2f} / {mm[t]['RMSE']:6.2f} "
    print(row)

## 5. Combinaciones de ensemble

Pruebo varias estrategias. Recuerda que **promediamos en unidades reales** (después del `inverse_transform`), no en escala normalizada — es lo mismo porque la transformación es lineal, pero es lo que se reporta.

- **Mean (top‑2)**: promedio de los 2 modelos con mejor `val_loss` durante entrenamiento (BiLSTM v2 y BiGRU).
- **Mean (top‑3)**: añade el siguiente, normalmente el Transformer grande.
- **Mean (all 5)**: promedio de los 5.
- **Weighted**: promedio ponderado por `1/val_loss` — los mejores cuentan más.
- **Median (all 5)**: mediana por elemento; robusta si un modelo se va muy lejos en algún ejemplo.

Determino el "top‑k" leyendo los `val_loss` reportados en los `metricas_*.json` (más justo que mirar el MAE de test, que sería *snooping*).

In [ ]:
# Leer val_loss reportado (best_val) de cada metricas_*.json
val_losses = {}
JSON_BY_NAME = {
    "BiLSTM v2":     "SALIDAS modelo 2/metricas_v2.json",
    "GRU":           "SALIDAS modelo GRU/metricas_gru.json",
    "Transformer":   "SALIDAS modelo Transformer/metricas_transformer.json",
    "BiGRU":         "SALIDAS modelo BiGRU/metricas_bigru.json",
    "Transf.Grande": "SALIDAS modelo Transformer Grande/metricas_transformer_grande.json",
}
# val_loss no se guarda explícitamente; uso mean_mae como proxy de calidad ordenadora
quality = {}
for name, path in JSON_BY_NAME.items():
    if name in preds and os.path.exists(path):
        d = json.load(open(path))
        quality[name] = d["mean_mae"]   # menor = mejor

ranked = sorted(quality, key=lambda n: quality[n])
print("Orden por mean_mae reportado (mejor -> peor):")
for n in ranked: print(f"  {n:<16} mean_mae={quality[n]:.3f}")

# Apilar predicciones en un tensor (n_modelos, n_test, 3) para combinarlas fácil
names = list(preds.keys())
stack = np.stack([preds[n] for n in names], axis=0)
def ens_mean(subset):  return stack[[names.index(n) for n in subset]].mean(axis=0)
def ens_median(subset):return np.median(stack[[names.index(n) for n in subset]], axis=0)
def ens_weighted(subset, weights):
    idx = [names.index(n) for n in subset]
    w = np.array(weights, dtype="float32"); w /= w.sum()
    return (stack[idx] * w[:, None, None]).sum(axis=0)

ensembles = {}
ensembles["Mean top-2"]    = ens_mean(ranked[:2])
ensembles["Mean top-3"]    = ens_mean(ranked[:3])
ensembles["Mean all 5"]    = ens_mean(ranked)
ensembles["Weighted all 5"]= ens_weighted(ranked, [1.0/quality[n] for n in ranked])
ensembles["Median all 5"]  = ens_median(ranked)

ens_metrics = {name: metrics_of(p) for name, p in ensembles.items()}

print("\nMÉTRICAS DE LOS ENSEMBLES — TEST")
print("="*70)
print(f"  {'Estrategia':<18}" + "".join(f"{t:>20}" for t in TARGET_COLS))
for name, mm in ens_metrics.items():
    row = f"  {name:<18}"
    for t in TARGET_COLS: row += f"   {mm[t]['MAE']:6.2f} / {mm[t]['RMSE']:6.2f} "
    print(row)

## 6. Comparativa final: individuales + ensembles + LSTM v1

Junto todo: los 5 modelos PyTorch, los 5 ensembles, y el **LSTM v1** (su MAE/RMSE leído del JSON, ya que sus pesos son `.keras` y no los tengo aquí). El "ganador" del ensemble es el que tenga menor `mean_mae`. Por experiencia, suele ser un mean top‑2 o top‑3 — los Transformer arrastran al ensemble si los incluyes y son peores.

In [ ]:
# Añadir LSTM v1 (solo lectura de su JSON)
v1_path = "outputs/results_lstm_original.json"
if os.path.exists(v1_path):
    d = json.load(open(v1_path))
    v1 = {t: {"MAE": d["targets"][t]["mae"], "RMSE": d["targets"][t]["rmse"]} for t in TARGET_COLS}
else:
    v1 = None

all_results = {}
if v1 is not None: all_results["LSTM v1"] = v1
for n in names: all_results[n] = indiv[n]
for n, mm in ens_metrics.items(): all_results[f"ENS: {n}"] = mm

# Tabla con mean_mae para ordenar
print("RANKING POR mean_mae (test, menor = mejor)")
print("="*60)
rank = sorted(all_results.items(),
              key=lambda kv: np.mean([kv[1][t]["MAE"] for t in TARGET_COLS]))
for i, (name, mm) in enumerate(rank, 1):
    mm_mae = np.mean([mm[t]["MAE"] for t in TARGET_COLS])
    flag = "  ⭐" if i == 1 else ""
    print(f"  {i:2d}. {name:<24} mean_mae={mm_mae:.3f}{flag}")

# Gráfico de barras MAE por target, con ensembles destacados
import matplotlib.cm as cm
labels = list(all_results.keys())
is_ens = [l.startswith("ENS:") for l in labels]
colors = ["#B71C1C" if e else "#1565C0" for e in is_ens]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Comparativa MAE — modelos individuales (azul) vs ensembles (rojo)", fontsize=13)
x = np.arange(len(labels))
for ax, t in zip(axes, TARGET_COLS):
    vals = [all_results[l][t]["MAE"] for l in labels]
    bars = ax.bar(x, vals, color=colors, edgecolor="white")
    bm = min(vals); ax.axhline(bm, color="green", linestyle="--", linewidth=1, alpha=0.6)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{t}", fontsize=11); ax.set_ylabel("MAE")
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
out_path = f"{OUT_DIR_PLOTS}/comparativa_ensemble.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight"); plt.show()
print(f"\nGuardado: {out_path}")

## 7. Guardar las métricas del mejor ensemble

Guardo en `SALIDAS modelo Ensemble/metricas_ensemble.json` la estrategia ganadora con el mismo esquema que el resto de notebooks, para que pueda integrarse en cualquier tabla comparativa posterior. También guardo las predicciones del mejor ensemble en `.npy` por si quieres usarlas para análisis adicionales (residuos, error por hora, etc.).

In [ ]:
# Mejor ensemble por mean_mae
best_name = min(ens_metrics, key=lambda n: np.mean([ens_metrics[n][t]["MAE"] for t in TARGET_COLS]))
best_mm   = ens_metrics[best_name]
mean_mae  = float(np.mean([best_mm[t]["MAE"]  for t in TARGET_COLS]))
mean_rmse = float(np.mean([best_mm[t]["RMSE"] for t in TARGET_COLS]))

results = {
    "model_name": f"ensemble_{best_name.lower().replace(' ', '_')}",
    "strategy":   best_name,
    "members":    ranked[:2] if "top-2" in best_name else (ranked[:3] if "top-3" in best_name else ranked),
    "targets": {t: {"mae": round(best_mm[t]["MAE"], 6), "rmse": round(best_mm[t]["RMSE"], 6)} for t in TARGET_COLS},
    "mean_mae":   round(mean_mae, 6),
    "mean_rmse":  round(mean_rmse, 6),
    "all_ensembles": {n: {t: {"mae": round(mm[t]["MAE"], 6), "rmse": round(mm[t]["RMSE"], 6)} for t in TARGET_COLS}
                       for n, mm in ens_metrics.items()},
    "individuals":   {n: {t: {"mae": round(mm[t]["MAE"], 6), "rmse": round(mm[t]["RMSE"], 6)} for t in TARGET_COLS}
                       for n, mm in indiv.items()},
}
json_path = os.path.join(OUT_DIR_PLOTS, "metricas_ensemble.json")
with open(json_path, "w") as f: json.dump(results, f, indent=2)

# Guardar predicciones del mejor ensemble
np.save(os.path.join(OUT_DIR_PLOTS, "preds_best_ensemble.npy"), ensembles[best_name])

print(f"Mejor ensemble: {best_name}  (mean_mae={mean_mae:.3f}, mean_rmse={mean_rmse:.3f})")
print(f"Métricas    -> {json_path}")
print(f"Predicciones-> {OUT_DIR_PLOTS}/preds_best_ensemble.npy  shape {ensembles[best_name].shape}")
print("\nFASE Ensemble completada.")